## 第 1 课：SM 架构、CUDA Core 与 Tensor Core

> 对应原文：Extra（基础知识参考）。这是进入 CuTe / CUTLASS 前必须补的硬件背景。

## 学习目标

搞清楚三件事：SM 里有哪些计算单元、它们怎么演进、GEMM 为什么必须依赖 Tensor Core。

## 1. SM 架构演进

SM（Streaming Multiprocessor）是 GPU 的基本计算单元。计算单元的演进：

- **Pascal**：Unified Int32 & FP32 Core（整数/浮点统一）；
- **Volta / Ampere / Hopper**：分离 Int32 和 FP32 单元，新增 FP64 单元；从 **Volta 起新增 Tensor Core**；
- **Blackwell**：FP32 与 Int32 重新统一，Tensor Core 升级到第五代。

![从 Volta 到 Blackwell 的 SM 架构演进](assets/figs/fig_02_从_Volta_到_Blackwell_的_SM_架构演进.png)

## 2. "CUDA Core" 是什么？

> "CUDA core" is a marketing term, not a technical term. —— NVIDIA 论坛

SM 架构里并没有写明哪些是 CUDA Core。一般把使用最频繁的 **FP32 计算单元**称为 CUDA Core。本系列中，**CUDA Core 指代除 Tensor Core 之外的所有计算单元**（Int、FP、SFU 等），它们负责标量/向量数值运算、地址计算、`blockIdx`/`threadIdx` 等辅助工作。

CUDA Core 也能算矩阵，但要按单个数值的粒度组织循环：

In [ ]:
__global__ void mm(float* A, float* B, float* C, int M, int N, int K) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < M && col < N) {
        float sum = 0.0f;
        for (int k = 0; k < K; ++k) {
            sum += A[row * K + k] * B[k * N + col];
        }
        C[row * N + col] = sum;
    }
}

高效 SGEMM 还需要考虑 thread hierarchy、memory hierarchy、block tile、shared memory、向量化访存——这也是本系列后面几课的主题。

## 3. Tensor Core：关注粒度从"数值"变成"矩阵"

Tensor Core 出现后，关注点从单个数值变为 **M×N 的单元矩阵**，编程时可以站在单元矩阵的角度编排计算。硬件指令的 DSA 化（Domain-Specific Architecture）凸显了张量运算的核心地位：最新的 Blackwell 上，甚至可以只在 **1 个线程**上完成所有 Tensor Core 的计算调度。

![Volta 架构下的单指令 Tensor Core 计算](assets/figs/fig_02_Volta_架构下的单指令_Tensor_Core_计算.png)

第一代 Tensor Core（Volta）单周期可计算 **4×4×4** 规模的 FP16 MMA，算力 4×4×4 = 128 FLOPs/cycle（原文图注为 2444，以原文为准）；此后每一代算力翻倍，Blackwell 第五代单 TC 算力达 2048 FLOPs/cycle。

理论算力公式：

In [ ]:
理论算力 = Tensor Core 时钟频率 × 单个 Tensor Core 算力 × SM 个数 × 单个 SM 中 Tensor Core 个数

（常用硬件性能指标见原文配图，B200 具体 spec 未公开，数据仅供参考）

![常用硬件性能指标](assets/figs/fig_05.png)

## 4. 涉及的两类 PTX 指令

- **CUDA Core 通用计算指令**：`add`、`sub`、`sin`、`cos`、`ex2` 等；
- **Tensor Core 张量指令**：与 SM 架构强相关，如 `mma`（Ampere）、`wgmma`（Hopper）、`tcgen05`（Blackwell）。

后续笔记会分析代码实际使用了哪些 PTX 指令。

## 你的学习路径提示

这一课不需要写代码。但要建立三个心智模型：

1. **内存层级**：GMEM（片外，容量大延迟高）→ SMEM（片内共享，几十~几百 KB）→ Register/RMEM（片内，延迟最低容量最小）；
2. **线程层级**：grid → block → warp（32 线程，SIMT 同时执行）→ thread；
3. **算力视角**：当今所有 GEMM 优化都围绕"最大化利用 Tensor Core"。

## 同时回答

1. 从 Volta 到 Blackwell，SM 计算单元经历了哪几个关键变化？（Int/FP 分离与统一、Tensor Core 代数）
2. 为什么说 "CUDA core" 是一个营销术语？本系列中它指代什么？
3. 理论算力为什么达不到？CUDA Core 和 Tensor Core 在指令层面（PTX）各有哪些代表性指令？

把答案发给我，我继续审查。